<a href="https://colab.research.google.com/github/jagadeeshdandu/NASSCOM-AI-FDP/blob/main/Day_5_Building_ML_Ready_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_loans(csv_path="loan_applications.csv", seed=23, verbose=False):
    """Realistic loan / credit-risk dataset for building an ML-ready pipeline.

    Built-in realism:
      - imbalanced target (default ~ 16%)
      - mixed numeric + categorical features
      - missing values, a few duplicate rows
      - a DELIBERATELY LEAKY column ('collection_calls') that is only known
        AFTER an account defaults — students must detect & drop it.
    """
    rng = np.random.default_rng(seed)
    N = 4000

    age = np.clip(rng.normal(40, 12, N), 21, 75).round().astype(int)
    income = np.clip(rng.lognormal(11.0, 0.45, N), 12000, None).round(-2)         # right-skewed
    employment_years = np.clip(rng.gamma(3, 2.2, N), 0, 40).round(1)
    credit_score = np.clip(rng.normal(680, 70, N), 300, 850).round().astype(int)
    loan_amount = np.clip(rng.lognormal(10.2, 0.5, N), 1000, None).round(-2)
    loan_term = rng.choice([12, 24, 36, 48, 60], N, p=[.12, .23, .33, .17, .15])
    num_existing_loans = rng.poisson(1.1, N)
    dti = np.clip(rng.normal(25, 10, N) + (loan_amount / (income + 1)) * 15, 2, 90).round(1)
    interest_rate = np.clip(14 - (credit_score - 680) / 35 + rng.normal(0, 1.2, N), 4, 28).round(2)
    home = rng.choice(["Rent", "Own", "Mortgage"], N, p=[.45, .20, .35])
    purpose = rng.choice(["Car", "Home", "Education", "Business", "Personal"],
                         N, p=[.22, .18, .15, .15, .30])
    region = rng.choice(["North", "South", "East", "West", "Central"],
                        N, p=[.24, .22, .18, .20, .16])
    prior_default = rng.choice(["Yes", "No"], N, p=[.14, .86])

    # ---- default risk (real signal) ----
    z = (-2.0
         - 0.012 * (credit_score - 680)
         + 0.035 * (dti - 28)
         + 0.06 * (interest_rate - 12)
         - 0.0000035 * (income - 60000)
         + 0.9 * (prior_default == "Yes")
         + 0.12 * num_existing_loans)
    p = 1 / (1 + np.exp(-z))
    default = (rng.random(N) < p).astype(int)

    # ---- LEAKY feature: collection calls happen only AFTER default ----
    collection_calls = np.where(default == 1, rng.poisson(6, N), rng.poisson(0.2, N))

    df = pd.DataFrame({
        "loan_id": [f"LN{i+1:05d}" for i in range(N)],
        "age": age, "annual_income": income, "employment_years": employment_years,
        "credit_score": credit_score, "loan_amount": loan_amount,
        "loan_term_months": loan_term, "num_existing_loans": num_existing_loans,
        "debt_to_income": dti, "interest_rate": interest_rate,
        "home_ownership": home, "loan_purpose": purpose, "region": region,
        "prior_default": prior_default,
        "collection_calls": collection_calls,            # <-- leakage trap
        "default": default,
    })

    # ---- messiness: missing values + a few duplicates ----
    for col, frac in [("annual_income", 0.05), ("employment_years", 0.06), ("home_ownership", 0.02)]:
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    df = pd.concat([df, df.sample(12, random_state=2)], ignore_index=True)

    df.to_csv(csv_path, index=False)
    if verbose:
        print("loans:", df.shape)
        print("default rate:", round(df["default"].mean(), 3))
        corr = df[["collection_calls", "credit_score", "debt_to_income", "default"]].corr()["default"]
        print("corr with default:\n", corr.round(3).to_string())
        print("duplicates:", int(df.duplicated().sum()),
              "| missing income:", int(df["annual_income"].isna().sum()))
    return df

if not os.path.exists('loan_applications.csv'):
    build_loans(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')


Generated dataset file.


In [2]:
import pandas as pd, numpy as np
df = pd.read_csv('loan_applications.csv')
print('shape:', df.shape)
print('default rate:', round(df['default'].mean(), 3))
df.head(3)

shape: (4012, 16)
default rate: 0.239


,loan_id,age,annual_income,employment_years,credit_score,loan_amount,loan_term_months,num_existing_loans,debt_to_income,interest_rate,home_ownership,loan_purpose,region,prior_default,collection_calls,default
0,LN00001,47,124000.0,8.3,757,18600.0,36,1,19.4,12.33,Own,Car,Central,No,1,0
1,LN00002,43,97200.0,5.0,677,26500.0,12,0,18.9,15.87,Own,Business,East,Yes,0,0
2,LN00003,39,119100.0,5.1,591,16900.0,48,2,37.3,18.65,Rent,Personal,West,No,0,0


In [3]:
# Quick clean (duplicates & a missingness check)

# -----------------------------------------------------------
# 🔹 1A. DROP DUPLICATES; NOTE MISSINGNESS (the pipeline will impute)
# -----------------------------------------------------------
print('duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('after drop:', df.shape)
print('\nmissing values:')
print(df.isna().sum()[lambda s: s > 0])

duplicate rows: 12
after drop: (4000, 16)

missing values:
annual_income       200
employment_years    240
home_ownership       80
dtype: int64


In [4]:
# Separate features (X) and target (y)

# -----------------------------------------------------------
# 🔹 2A. y = what we predict; X = what we're allowed to use
# -----------------------------------------------------------
y = df['default']
X = df.drop(columns=['default', 'loan_id'])   # drop the target and the ID
print('X:', X.shape, '| y:', y.shape)
print('feature columns:', list(X.columns))

X: (4000, 14) | y: (4000,)
feature columns: ['age', 'annual_income', 'employment_years', 'credit_score', 'loan_amount', 'loan_term_months', 'num_existing_loans', 'debt_to_income', 'interest_rate', 'home_ownership', 'loan_purpose', 'region', 'prior_default', 'collection_calls']


In [5]:
# Leakage hunt — the cardinal sin

# -----------------------------------------------------------
# 🔹 3A. CORRELATION OF EACH NUMERIC FEATURE WITH THE TARGET
# -----------------------------------------------------------
num_cols = X.select_dtypes('number').columns
corr_y = X[num_cols].corrwith(y).abs().sort_values(ascending=False)
print('Absolute correlation with default:')
print(corr_y.round(3))
print('\nThat top value is suspiciously high — investigate it.')

Absolute correlation with default:
collection_calls      0.893
credit_score          0.321
interest_rate         0.295
debt_to_income        0.170
annual_income         0.094
num_existing_loans    0.083
loan_term_months      0.037
loan_amount           0.033
age                   0.013
employment_years      0.006
dtype: float64

That top value is suspiciously high — investigate it.


In [6]:
# -----------------------------------------------------------
# 🔹 3B. WHY 'collection_calls' IS LEAKAGE
# -----------------------------------------------------------
print(df.groupby('default')['collection_calls'].mean().round(2))
print('\nCollection calls only happen AFTER a loan starts defaulting —')
print("we would NOT know this at application time. It's leakage. Drop it.")
X = X.drop(columns=['collection_calls'])
print('features now:', X.shape[1])

default
0    0.2
1    6.1
Name: collection_calls, dtype: float64

Collection calls only happen AFTER a loan starts defaulting —
we would NOT know this at application time. It's leakage. Drop it.
features now: 13


In [8]:
# This is the most important exercise in the lab.

# 1. Build a quick numeric-only logistic-regression pipeline (impute + scale + model).
# 2. Get its 5-fold CV accuracy using X_leaky = df.drop(columns=['default','loan_id']).select_dtypes('number') (which still contains collection_calls).
# 3. Get the CV accuracy on X.select_dtypes('number') (leakage removed).
# 4. In a comment, report both numbers and explain the gap.


from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe_q = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                       LogisticRegression(max_iter=1000))

In [9]:
X_leaky = df.drop(columns=['default','loan_id']).select_dtypes('number')
leaky_cv_accuracy = cross_val_score(pipe_q, X_leaky, y, cv=5, scoring='accuracy')
print(f"Leaky CV Accuracy: {leaky_cv_accuracy.mean():.3f} (+/- {leaky_cv_accuracy.std():.3f})")

Leaky CV Accuracy: 0.988 (+/- 0.003)


In [10]:
non_leaky_cv_accuracy = cross_val_score(pipe_q, X.select_dtypes('number'), y, cv=5, scoring='accuracy')
print(f"Non-leaky CV Accuracy: {non_leaky_cv_accuracy.mean():.3f} (+/- {non_leaky_cv_accuracy.std():.3f})")

Non-leaky CV Accuracy: 0.773 (+/- 0.013)


The significant drop in accuracy after removing `collection_calls` demonstrates the impact of data leakage. The `collection_calls` feature was highly correlated with the target variable `default` because collection calls only occur *after* a loan has defaulted. This information would not be available at the time of making a prediction for a new loan application, making it a leaky feature. Including it in the training data leads to an overly optimistic and unrealistic performance metric.

In [11]:
X.select_dtypes('number')

,age,annual_income,employment_years,credit_score,loan_amount,loan_term_months,num_existing_loans,debt_to_income,interest_rate
0,47,124000.0,8.3,757,18600.0,36,1,19.4,12.33
1,43,97200.0,5.0,677,26500.0,12,0,18.9,15.87
2,39,119100.0,5.1,591,16900.0,48,2,37.3,18.65
3,21,44300.0,5.8,661,31800.0,36,0,34.5,13.62
4,45,123900.0,2.3,674,23500.0,36,2,20.7,14.00
...,...,...,...,...,...,...,...,...,...
3995,31,27800.0,9.1,731,44900.0,48,2,45.0,13.78
3996,43,63500.0,8.3,590,36700.0,24,2,34.4,16.15
3997,36,48800.0,NaN,692,52400.0,60,0,47.6,13.49
3998,28,109800.0,4.0,635,17700.0,24,1,37.3,15.39


The 5-fold cross-validation accuracy on the non-leaky dataset (X.select_dtypes('number')) was calculated as 0.773 (+/- 0.013). This was already provided in the previous step, along with an explanation of the difference between the leaky and non-leaky models.

Leaky CV Accuracy: 0.988 (+/- 0.003) Non-leaky CV Accuracy: 0.773 (+/- 0.013)

This significant difference highlights the data leakage from the collection_calls feature.

In [12]:
# Stratified train/test split

# -----------------------------------------------------------
# 🔹 4A. SPLIT FIRST — and stratify because the target is imbalanced
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('train:', X_train.shape, '| test:', X_test.shape)
print('default rate  train / test:',
      round(y_train.mean(), 3), '/', round(y_test.mean(), 3),
      ' <- preserved by stratify')


train: (3200, 13) | test: (800, 13)
default rate  train / test: 0.239 / 0.239  <- preserved by stratify


### Non-stratified train/test split

In [14]:
# Make a non-stratified split (same test_size, random_state) and print the train/test default rates.

X_train_non_strat, X_test_non_strat, y_train_non_strat, y_test_non_strat = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('train:', X_train_non_strat.shape, '| test:', X_test_non_strat.shape)
print('default rate  train / test (non-stratified):',
      round(y_train_non_strat.mean(), 3), '/', round(y_test_non_strat.mean(), 3),
      ' <- may not be preserved without stratify')

train: (3200, 13) | test: (800, 13)
default rate  train / test (non-stratified): 0.242 / 0.229  <- may not be preserved without stratify


### Comparison and Importance of Stratification

**Stratified Split Default Rates (train / test):** 0.239 / 0.239
**Non-stratified Split Default Rates (train / test):** 0.242 / 0.229

As observed, the stratified split successfully maintained almost identical default rates in both the training and testing sets. In contrast, the non-stratified split shows a slight difference (0.242 in train vs. 0.229 in test).

Stratifying matters more as a class gets rarer because, without it, there's an increased risk that a random split could result in a training or testing set with very few, or even zero, instances of the minority class. This would severely impact the model's ability to learn about or evaluate its performance on that rare class, leading to a biased or unreliable model. Stratification ensures that each split is a representative sample of the entire dataset's class distribution.

In [16]:
# Handling class imbalance

# -----------------------------------------------------------
# 🔹 5A. WHY ACCURACY LIES ON IMBALANCED DATA
# -----------------------------------------------------------
majority = 1 - y.mean()
print(f'Always predicting "no default" scores {majority:.1%} accuracy —')
print('yet catches ZERO defaulters. Accuracy alone is misleading here.')
print('Fixes: stratify (done), class_weight="balanced", or resampling.')

Always predicting "no default" scores 76.1% accuracy —
yet catches ZERO defaulters. Accuracy alone is misleading here.
Fixes: stratify (done), class_weight="balanced", or resampling.


In [ ]:
# EXERCISE 5 — Balanced vs default model
# 1. Cross-validate a logistic regression with class_weight='balanced' and score with scoring='recall'.
# 2. Do the same without class weights.
# 3. In a comment, say which catches more defaulters (higher recall).

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat)])

In [18]:
# balanced model recall (cv=5, scoring='recall')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
pre = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat)])

pipe_balanced = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight='balanced'))
balanced_recall_cv = cross_val_score(pipe_balanced, X_train, y_train, cv=5, scoring='recall')
print(f"Balanced Model Recall (CV): {balanced_recall_cv.mean():.3f} (+/- {balanced_recall_cv.std():.3f})")

Balanced Model Recall (CV): 0.701 (+/- 0.051)


In [19]:
pipe_default = make_pipeline(pre, LogisticRegression(max_iter=1000))
default_recall_cv = cross_val_score(pipe_default, X_train, y_train, cv=5, scoring='recall')
print(f"Default Model Recall (CV): {default_recall_cv.mean():.3f} (+/- {default_recall_cv.std():.3f})")

Default Model Recall (CV): 0.257 (+/- 0.013)


In [20]:
class_weight='balanced'

To confirm, the model with class_weight='balanced' achieved a recall of 0.701 (+/- 0.051), while the model without class weights had a recall of 0.257 (+/- 0.013). This clearly shows that the model with class_weight='balanced' is significantly better at catching defaulters (it has a much higher recall).

Based on the cross-validation results, the model with class_weight='balanced' achieved a recall of 0.701, while the model without class weights had a recall of 0.257. Therefore, the model with class_weight='balanced' catches more defaulters.

In [21]:
# The leak-free preprocessing pipeline

# -----------------------------------------------------------
# 🔹 6A. ColumnTransformer + model, fitted on TRAIN ONLY
# -----------------------------------------------------------
from sklearn.pipeline import Pipeline
clf = Pipeline([('prep', pre),
                ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
clf.fit(X_train, y_train)              # every transformer learns from train only
print('test accuracy:', round(clf.score(X_test, y_test), 3))
from sklearn.metrics import recall_score
print('test recall (default class):',
      round(recall_score(y_test, clf.predict(X_test)), 3))

test accuracy: 0.71
test recall (default class): 0.654


In [22]:
# Cross-validation for a stable estimate

# -----------------------------------------------------------
# 🔹 7A. STRATIFIED 5-FOLD CV (mean ± spread)
# -----------------------------------------------------------
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
print('ROC-AUC per fold:', scores.round(3))
print(f'mean {scores.mean():.3f}  ±  {scores.std():.3f}')

ROC-AUC per fold: [0.795 0.792 0.758 0.75  0.703]
mean 0.760  ±  0.034


In [24]:
# -----------------------------------------------------------
# 🔹 7B. STRATIFIED 5-FOLD CV (mean ± spread) with leakage
# -----------------------------------------------------------

# Redefine numeric columns for X_leaky
num_leaky = X_leaky.select_dtypes('number').columns.tolist()

# Redefine the ColumnTransformer for X_leaky to only handle numeric columns,
# as X_leaky (defined previously) only contains numeric features.
pre_leaky = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_leaky)
])

# Create a new pipeline with the leaky preprocessor and balanced Logistic Regression
clf_leaky = Pipeline([('prep', pre_leaky),
                      ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])

# Perform cross-validation on X_leaky
scores_leaky = cross_val_score(clf_leaky, X_leaky, y, cv=5, scoring='roc_auc')
print('ROC-AUC per fold (with leakage):', scores_leaky.round(3))
print(f'mean {scores_leaky.mean():.3f}  ±  {scores_leaky.std():.3f}')

ROC-AUC per fold (with leakage): [0.996 0.999 0.999 0.992 0.998]
mean 0.997  ±  0.003


Let's directly compare the ROC-AUC scores:

Leak-Free Model (from cell wtVK0P1sGBk6): ROC-AUC of 0.760 (+/- 0.034)
Leaky Model (including collection_calls): ROC-AUC of 0.997 (+/- 0.003)
The significantly higher ROC-AUC in the leaky model (0.997 vs. 0.760) vividly demonstrates how data leakage can make a model appear to perform exceptionally well when, in reality, it's leveraging information that wouldn't be available at the time of prediction. This inflated score is misleading and highlights the critical importance of identifying and removing leaky features.

In [25]:
# Final ML-readiness check

# -----------------------------------------------------------
# 🔹 8A. GATE CHECKS — assert the dataset is truly ready
# -----------------------------------------------------------
checks = {
    'no leaky column': 'collection_calls' not in X.columns,
    'X and y aligned': len(X) == len(y),
    'target is binary': set(y.unique()) == {0, 1},
    'split is stratified': abs(y_train.mean() - y_test.mean()) < 0.02,
    'reproducible seed used': True,
}
for k, v in checks.items():
    print(('PASS' if v else 'FAIL'), '-', k)
print('\nReady for modelling:', all(checks.values()))

PASS - no leaky column
PASS - X and y aligned
PASS - target is binary
PASS - split is stratified
PASS - reproducible seed used

Ready for modelling: True


In [29]:
# Add a check that no feature column is missing after the pipeline transforms the training data

not np.isnan(clf.named_steps['prep'].transform(X_train)).any()

True